# Z 四等分专家 vs 单模型 · PSNR vs CR（60s）

**场**：`dark_matter_density` · `baryon_density` · `temperature`  
**rel**：`1e-5` … `6e-4`（与 `model.ipynb` 一致）  
**对比（均 60s 训练预算）**：
- `single_60`：1 个 Micro-UNet，全 z，`max_train_time=60s`
- `shard4_hetero_60`：4 专家异构 `bg_h`，每 shard `60s`（并行时墙钟≈60s）
- `shard4_uniform_60`：4 专家同 `bg_h`，每 shard `60s`

**参数预算**（`USE_PER_SHARD_BUDGET`）：`True` 时每个 shard/GPU 最多 `PARAM_BUDGET_PER_SHARD`（默认 4000，合计约 16k）；`False` 时 4 shard **共享** `PARAM_BUDGET_ENSEMBLE_TOTAL`（旧 4k 合计）。`bg_h` 由 `build_shard_plan` 自动选取，勿在 `EXPERIMENTS` 里写死。

**4 专家推理**：`infer_shard_ensemble_blend` — 各 shard 在 `[z0-O, z1+O)` 上推理，z 向 Hann 权重融合 AI 残差（`SHARD_OVERLAP` 可调），减轻硬拼接接缝。

图：**每场一张 PSNR vs effective CR**（含 SZ3 Xp 基线）。

In [ ]:
import random
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

sys.path.append("/home/sam/Halo_Finder/Final_design/base_script")

from config_io import load_multifield_from_disk
from experiment import build_bg_only_cfg, estimate_bg_model_param_bytes
from bg_stage import run_bg_inference, train_bg_only
from bg_shard import (
    z_quad_shard_bounds,
    build_shard_plan,
    crop_multifield_zyx,
    infer_shard_ensemble_blend,
    pick_bg_h_under_budget,
    train_shards_parallel,
    reload_shard_model,
    _build_shard_cfg,
)


def _global_diag(x_true, x_hat):
    """Global reconstruction metrics (replaces the old ROI diagnostics)."""
    x_true = np.asarray(x_true); x_hat = np.asarray(x_hat)
    dr = float(x_true.max() - x_true.min()) or 1.0
    mse = float(np.mean((x_true - x_hat) ** 2))
    psnr = 20 * np.log10(dr) - 10 * np.log10(mse + 1e-12) if mse > 0 else 100.0
    max_err = float(np.max(np.abs(x_true - x_hat)))
    return {"psnr": psnr, "max_err": max_err}

pysz_path = r"/home/sam/Data_Compression/SZ3/tools/pysz"
if pysz_path not in sys.path:
    sys.path.append(pysz_path)
from pysz import SZ


def set_seed(seed=17):
    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


NUM_GPUS = int(torch.cuda.device_count()) if torch.cuda.is_available() else 0
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"cuda devices: {NUM_GPUS} | device: {device}")

In [ ]:
base_path = Path(r"/home/sam/Halo_Finder/halo_finder_v1/SDRBENCH-EXASKY-NYX-512x512x512/origin").resolve().as_posix() + "/"
sz_lib_path = r"/home/sam/Data_Compression/SZ3/build/lib64/libSZ3c.so"
data_shape = (512, 512, 512)

FIELD_FILES = [
    "dark_matter_density.f32", "velocity_z.f32", "baryon_density.f32",
    "temperature.f32", "velocity_x.f32", "velocity_y.f32",
]
TARGET_STEMS = ["dark_matter_density", "baryon_density", "temperature"]
FIELD_LABEL = {
    "dark_matter_density": "DM",
    "baryon_density": "BD",
    "temperature": "T",
}

REL_SETTINGS = [
    ("r0", 1e-4), ("r1", 2e-4), ("r2", 3e-4), ("r3", 4e-4),
    ("r4", 5e-4), ("r5", 6e-4), ("r6", 1e-5),
]
REL_PROBE = 1e-4

sz_engine = SZ(sz_lib_path)
D, H, W = data_shape
SHARD_BOUNDS = z_quad_shard_bounds(D, 4)
SHARD_OVERLAP = 16  # z 向 overlap；0 则退化为硬拼接
SHARD_BLEND = "hann"
SHARD_WEIGHT_FLOOR = 0.05


def rel_sz_suffix(rel_err):
    return f"{float(rel_err):.0e}".replace("+", "")


def load_field_data(target_stem, rel_probe=REL_PROBE):
    fname = f"{target_stem}.f32"
    gt_path = base_path + fname
    aux_paths = [base_path + f for f in FIELD_FILES if f != fname]
    sz_bin = base_path + Path(fname).stem + "_rel" + rel_sz_suffix(rel_probe) + ".sz"
    if not Path(sz_bin).is_file():
        vol = np.fromfile(gt_path, dtype=np.float32).reshape(data_shape)
        Path(sz_bin).write_bytes(sz_engine.compress(vol, 1, 0, float(rel_probe), 0)[0])
    Xs, Xps = load_multifield_from_disk(
        gt_path=gt_path, aux_paths=aux_paths, sz_bin_path=sz_bin,
        data_shape=data_shape, pysz_path=pysz_path, sz_lib_path=sz_lib_path,
    )
    gt_target = np.asarray(Xs[0], np.float32)
    aux_fields = [np.asarray(f, np.float32) for f in Xs[1:]]

    def build_Xps_for_rel(rel_err):
        b, _cr = sz_engine.compress(gt_target, 1, 0, float(rel_err), 0)
        x_lq = sz_engine.decompress(b, gt_target.shape, np.float32)
        return [x_lq] + aux_fields, float(_cr), int(len(b))

    return {
        "target_stem": target_stem,
        "field_label": FIELD_LABEL.get(target_stem, target_stem[:8]),
        "Xs": Xs,
        "gt_target": gt_target,
        "build_Xps_for_rel": build_Xps_for_rel,
        "n_fields": len(Xs),
        "original_target_bytes": int(gt_target.nbytes),
    }


set_seed(17)
print("TARGET_STEMS:", TARGET_STEMS)

print("\n" + "="*50)
print("SZ3 Compression Ratios")
print("="*50)
for stem in TARGET_STEMS:
    field_name = FIELD_LABEL.get(stem, stem)
    print(f"\n[{field_name}]")
    
    # 针对当前物理场加载数据上下文
    ctx = load_field_data(stem)
    
    # 遍历所有的 rel_err 设置
    for r_name, r_val in REL_SETTINGS:
        # 调用自带的压缩函数获取压缩率 cr
        _, cr, b_len = ctx["build_Xps_for_rel"](r_val)
        print(f"  - {r_name} (rel={r_val:.0e}): CR = {cr:>6.2f}  |  Size = {b_len / 1024 / 1024:>6.2f} MB")

In [ ]:
BG_LR = 1e-3
BG_ARCH = "spatial"
BG_PATCH = 512
MODEL_DTYPE_BYTES = 2

# 参数预算：USE_PER_SHARD_BUDGET=True → 每张卡/每个 shard 最多 PARAM_BUDGET_PER_SHARD（4 卡合计约 4×）
# False → 旧行为，4 个 shard 共享 PARAM_BUDGET_ENSEMBLE_TOTAL
USE_PER_SHARD_BUDGET = True
PARAM_BUDGET_PER_SHARD = 30000
PARAM_BUDGET_ENSEMBLE_TOTAL = 4000

_h_single, _p_single = pick_bg_h_under_budget(
    PARAM_BUDGET_PER_SHARD if USE_PER_SHARD_BUDGET else PARAM_BUDGET_ENSEMBLE_TOTAL,
    shape=data_shape,
    n_fields=6,
)
SINGLE_BG_H = int(_h_single)  # 与 per-shard 预算对齐；可手动覆盖为固定 h
print(f"single bg_h={SINGLE_BG_H} (~{_p_single} params) | per_shard_budget={USE_PER_SHARD_BUDGET}")

TRAIN_TIME_S = 60.0
EPOCHS_CAP = 200

EXPERIMENTS = {
    "single_60": {
        "kind": "single",
        "label": "single | 60s",
        "max_train_time": TRAIN_TIME_S,
        "bg_h": SINGLE_BG_H,
        "color": "#4c72b0",
        "marker": "o",
    },
    "shard4_hetero_60": {
        "kind": "shard",
        "heterogeneous": True,
        "label": "4 experts hetero | 60s/shard",
        "max_train_time": TRAIN_TIME_S,
        "color": "#55a868",
        "marker": "s",
    },
    "shard4_uniform_60": {
        "kind": "shard",
        "heterogeneous": False,
        "label": "4 experts uniform | 60s/shard",
        "max_train_time": TRAIN_TIME_S,
        "color": "#c44e52",
        "marker": "D",
    },
}

CR_SWEEP_EXPERIMENTS = ["single_60", "shard4_hetero_60", "shard4_uniform_60"]
PARALLEL_SHARDS = True

# 快速试跑可缩小：
# SWEEP_REL_LIST = REL_SETTINGS[:2]
# TARGET_STEMS_CR = ["dark_matter_density"]
SWEEP_REL_LIST = REL_SETTINGS
TARGET_STEMS_CR = list(TARGET_STEMS)


def psnr_np(x_true, x_hat):
    x_true, x_hat = np.asarray(x_true, np.float64), np.asarray(x_hat, np.float64)
    mse = float(np.mean((x_true - x_hat) ** 2))
    dr = float(np.max(x_true) - np.min(x_true))
    if mse <= 0:
        return float("inf")
    return float(20.0 * np.log10(max(dr, 1e-12) / np.sqrt(mse)))


def model_param_bytes(model):
    return int(sum(p.numel() for p in model.parameters()) * MODEL_DTYPE_BYTES)


def effective_cr(sz3_bytes, nn_bytes, original_bytes):
    return float(original_bytes) / max(float(sz3_bytes) + float(nn_bytes), 1.0)


def build_cfg_global(ctx, rel_err, max_train_time, bg_h, steps_per_epoch, log_prefix=""):
    Xs = ctx["Xs"]
    Xps_list = ctx["Xps_list"]
    cfg = build_bg_only_cfg(
        X_target=Xs[0],
        Xps=Xps_list,
        max_train_time=float(max_train_time),
        bg_h=int(bg_h),
        roi_h=4,
        epochs=int(EPOCHS_CAP),
        steps_per_epoch=int(steps_per_epoch),
        bg_patch_size=int(BG_PATCH),
        bg_batch=1,
        lr=float(BG_LR),
    )
    cfg.bg_arch = BG_ARCH
    cfg.bg_split_mode = "three"
    cfg.bg_split_bands = True
    cfg.bg_sample_mode = "sequential"
    cfg.bg_log_prefix = str(log_prefix)
    cfg.bg_input_norm = "global"
    cfg.bg_residual_norm = "global"
    cfg.rel_err = float(rel_err)
    return cfg


def train_one_shard(ctx, shard, rel_err, max_train_time):
    Xs, Xps_list = ctx["Xs"], ctx["Xps_list"]
    z0, z1 = shard["z0"], shard["z1"]
    bg_h = shard["bg_h"]
    sid = shard["shard_id"]
    Xs_s, Xps_s = crop_multifield_zyx(Xs, Xps_list, z0, z1)
    cfg = build_cfg_global(
        ctx, rel_err, max_train_time, bg_h,
        steps_per_epoch=z1 - z0,
        log_prefix=f"{ctx['field_label']}|s{sid}_h{bg_h}",
    )
    dev = torch.device(f"cuda:{sid % max(NUM_GPUS, 1)}" if torch.cuda.is_available() else "cpu")
    first = [True]

    def evaluator(m):
        if first[0]:
            first[0] = False
            m2 = _global_diag(Xs[0], Xps_list[0])
            return m2["psnr"], m2["max_err"]
        xh = infer_shard_ensemble_blend(
            [m], [cfg], Xs, Xps_list, [(z0, z1)], rel_err,
            overlap=SHARD_OVERLAP, blend=SHARD_BLEND, weight_floor=SHARD_WEIGHT_FLOOR,
        )
        m2 = _global_diag(Xs[0], xh)
        return m2["psnr"], m2["max_err"]

    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    t0 = time.perf_counter()
    model, hist = train_bg_only(Xs=Xs_s, Xps=Xps_s, device=dev, cfg=cfg, evaluator=evaluator)
    wall = time.perf_counter() - t0
    _ew = list(hist.get("epoch_wall") or [])
    return model, cfg, wall, len(hist.get("loss") or []), (float(np.mean(_ew)) if _ew else float("nan"))


def run_experiment(exp_id, ctx, rel_err):
    spec = EXPERIMENTS[exp_id]
    rel_err = float(rel_err)
    Xs = ctx["Xs"]
    gt_target = ctx["gt_target"]
    orig_b = ctx["original_target_bytes"]
    stem = ctx["target_stem"]
    flab = ctx["field_label"]

    Xps_list, _, sz3_n = ctx["build_Xps_for_rel"](rel_err)
    Xps_list = [np.asarray(f, np.float32) for f in Xps_list]
    ctx = {**ctx, "Xps_list": Xps_list}

    print(f"\n{'='*64}\n  [{flab}] {exp_id}: {spec['label']} | rel={rel_err:.0e}\n{'='*64}")

    if spec["kind"] == "single":
        set_seed(17)
        cfg = build_cfg_global(
            ctx, rel_err, spec["max_train_time"], spec["bg_h"],
            steps_per_epoch=D, log_prefix=f"{flab}|{exp_id}",
        )
        first = [True]

        def evaluator(m):
            if first[0]:
                first[0] = False
                m2 = _global_diag(Xs[0], Xps_list[0])
                return m2["psnr"], m2["max_err"]
            xh = run_bg_inference(m, Xs, Xps_list, cfg, rel_err)
            m2 = _global_diag(Xs[0], xh)
            return m2["psnr"], m2["max_err"]

        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        t0 = time.perf_counter()
        model, hist = train_bg_only(Xs=Xs, Xps=Xps_list, device=device, cfg=cfg, evaluator=evaluator)
        wall = time.perf_counter() - t0
        x_hat = run_bg_inference(model, Xs, Xps_list, cfg, rel_err)
        nn_b = model_param_bytes(model)
        return {
            "target_stem": stem,
            "field_label": flab,
            "rel_err": rel_err,
            "exp": exp_id,
            "exp_label": spec["label"],
            "psnr_512": float(psnr_np(gt_target, x_hat)),
            "wall_s": float(wall),
            "nn_bytes": int(nn_b),
            "sz3_bytes": int(sz3_n),
            "effective_cr": effective_cr(sz3_n, nn_b, orig_b),
            "train_epochs": int(len(hist.get("loss") or [])),
            "history_psnr": hist.get("psnr"),
            "history_time": hist.get("time"),
        }

    plan_raw = build_shard_plan(
        gt_target, Xps_list[0], D,
        total_param_budget=PARAM_BUDGET_ENSEMBLE_TOTAL,
        per_shard_param_budget=PARAM_BUDGET_PER_SHARD if USE_PER_SHARD_BUDGET else None,
        heterogeneous=bool(spec["heterogeneous"]),
    )
    shards = [p for p in plan_raw if "shard_id" in p]
    _plan_meta = plan_raw[-1] if plan_raw else {}
    print(
        f"  shard plan: per_shard_budget={USE_PER_SHARD_BUDGET} "
        f"total_nn_params≈{_plan_meta.get('total_params', '?')} "
        f"hetero={_plan_meta.get('heterogeneous')}"
    )
    for sh in shards:
        print(f"    s{sh['shard_id']} z=[{sh['z0']}:{sh['z1']}) h={sh['bg_h']} n_params={sh['n_params']}")

    if PARALLEL_SHARDS and NUM_GPUS >= len(shards):
        t0 = time.perf_counter()
        worker_out = train_shards_parallel(
            shards, Xs, Xps_list, rel_err, spec["max_train_time"],
            epochs=int(EPOCHS_CAP), n_gpus=NUM_GPUS,
        )
        wall_parallel = time.perf_counter() - t0
        models, cfgs = [], []
        for wo in sorted(worker_out, key=lambda x: x["shard_id"]):
            cfg = _build_shard_cfg(
                Xs, Xps_list, shards[wo["shard_id"]], rel_err,
                spec["max_train_time"], epochs=int(EPOCHS_CAP),
            )
            m = reload_shard_model(
                wo["state_dict"], cfg, shape=Xs[0].shape,
                n_fields=ctx["n_fields"], device=device,
            )
            models.append(m)
            cfgs.append(cfg)
            print(
                f"  shard {wo['shard_id']} z=[{wo['z0']}:{wo['z1']}) h={wo['bg_h']} "
                f"wall={wo['wall_s']:.1f}s epochs={wo['train_epochs']}"
            )
        wall_max = float(wall_parallel)
        _n_ep = int(worker_out[0]["train_epochs"]) if worker_out else 0
    else:
        models, cfgs, walls, n_eps = [], [], [], []
        for shard in shards:
            set_seed(17 + int(shard["shard_id"]))
            m, c, w, n_ep, _ = train_one_shard(ctx, shard, rel_err, spec["max_train_time"])
            models.append(m)
            cfgs.append(c)
            walls.append(w)
            n_eps.append(n_ep)
            print(
                f"  shard {shard['shard_id']} z=[{shard['z0']}:{shard['z1']}) h={shard['bg_h']} "
                f"wall={w:.1f}s epochs={n_ep}"
            )
        wall_max = float(max(walls))
        _n_ep = int(n_eps[0]) if n_eps else 0

    x_hat = infer_shard_ensemble_blend(
        models, cfgs, Xs, Xps_list, SHARD_BOUNDS, rel_err,
        overlap=SHARD_OVERLAP, blend=SHARD_BLEND, weight_floor=SHARD_WEIGHT_FLOOR,
    )
    nn_b = sum(model_param_bytes(m) for m in models)
    return {
        "target_stem": stem,
        "field_label": flab,
        "rel_err": rel_err,
        "exp": exp_id,
        "exp_label": spec["label"],
        "psnr_512": float(psnr_np(gt_target, x_hat)),
        "wall_s": wall_max,
        "nn_bytes": int(nn_b),
        "sz3_bytes": int(sz3_n),
        "effective_cr": effective_cr(sz3_n, nn_b, orig_b),
        "train_epochs": _n_ep,
        "n_shards": len(shards),

        "history_psnr": None, 
        "history_time": None, 
    }

In [ ]:
# ---- CR sweep: 3 fields × 7 rel × {single, hetero experts, uniform experts} ----
CR_ROWS = []
XP_ROWS = []

for target_stem in TARGET_STEMS_CR:
    ctx_base = load_field_data(target_stem)
    for tag, rel_err in SWEEP_REL_LIST:
        Xps_probe, _, b_probe = ctx_base["build_Xps_for_rel"](rel_err)
        XP_ROWS.append({
            "target_stem": target_stem,
            "field_label": ctx_base["field_label"],
            "rel_err": float(rel_err),
            "rel_tag": tag,
            "psnr_xp": float(psnr_np(ctx_base["gt_target"], Xps_probe[0])),
            "effective_cr_xp": float(ctx_base["original_target_bytes"] / max(int(b_probe), 1)),
            "sz3_bytes": int(b_probe),
        })

        for exp_id in CR_SWEEP_EXPERIMENTS:
            ctx = {**ctx_base}
            try:
                row = run_experiment(exp_id, ctx, rel_err)
                row["rel_tag"] = tag
                CR_ROWS.append(row)
            except RuntimeError as e:
                if "out of memory" in str(e).lower() or "cuda" in str(e).lower():
                    print(f"[OOM skip] {target_stem} {exp_id} rel={rel_err:.0e}: {e}")
                    if torch.cuda.is_available():
                        torch.cuda.empty_cache()
                else:
                    raise

df_cr = pd.DataFrame(CR_ROWS)
df_xp = pd.DataFrame(XP_ROWS).sort_values(["target_stem", "effective_cr_xp"])
globals()["SHARD_CR_DF"] = df_cr

display(df_cr.sort_values(["target_stem", "rel_err", "exp"]))
print(f"runs: {len(df_cr)} | fields: {df_cr['target_stem'].nunique() if len(df_cr) else 0}")

In [ ]:
# ---- PSNR vs effective CR（DM / BD / T，60s single + experts）----
if df_cr.empty:
    raise RuntimeError("df_cr 为空：先跑上一格 CR sweep")

n_fields = len(TARGET_STEMS_CR)
fig, axes = plt.subplots(1, n_fields, figsize=(5.5 * n_fields, 5), dpi=140, squeeze=False)
axes = axes.ravel()

for ax, stem in zip(axes, TARGET_STEMS_CR):
    flab = FIELD_LABEL.get(stem, stem)
    sub = df_cr[df_cr["target_stem"] == stem]
    xp = df_xp[df_xp["target_stem"] == stem].sort_values("effective_cr_xp")

    if not xp.empty:
        ax.plot(
            xp["effective_cr_xp"], xp["psnr_xp"],
            "k--", lw=1.8, ms=0, label="SZ3 Xp only",
        )

    for exp_id in CR_SWEEP_EXPERIMENTS:
        spec = EXPERIMENTS[exp_id]
        g = sub[sub["exp"] == exp_id].sort_values("effective_cr")
        if g.empty:
            continue
        ax.plot(
            g["effective_cr"], g["psnr_512"],
            color=spec.get("color", None),
            marker=spec.get("marker", "o"),
            lw=2.0, ms=7, label=spec["label"],
        )
        for _, row in g.iterrows():
            ax.annotate(
                f"{row['rel_err']:.0e}",
                (row["effective_cr"], row["psnr_512"]),
                textcoords="offset points", xytext=(3, 3), fontsize=6,
                color=spec.get("color", "gray"),
            )

    ax.set_title(f"{flab} — PSNR vs CR (train {int(TRAIN_TIME_S)}s)")
    ax.set_xlabel("effective CR = original / (SZ3 + NN)")
    ax.set_ylabel("PSNR @ 512³ (dB)")
    ax.grid(True, alpha=0.35)
    ax.legend(loc="lower left", fontsize=8)

fig.suptitle("Single model vs 4-shard experts (60s budget)", y=1.02, fontsize=13)
fig.tight_layout()
plt.show()

# 汇总表：每场 best PSNR per method
best_tbl = (
    df_cr.groupby(["target_stem", "field_label", "exp", "exp_label"], as_index=False)
    .agg(best_psnr=("psnr_512", "max"), mean_cr=("effective_cr", "mean"))
    .sort_values(["target_stem", "best_psnr"], ascending=[True, False])
)
display(best_tbl)

In [ ]:
# 可选：单 rel 快速对比（probe）
PROBE_EXPERIMENTS = CR_SWEEP_EXPERIMENTS
ctx_probe = load_field_data("dark_matter_density")
probe_rows = []
for exp_id in PROBE_EXPERIMENTS:
    probe_rows.append(run_experiment(exp_id, ctx_probe, REL_PROBE))
df_probe = pd.DataFrame(probe_rows)
display(df_probe[["exp", "psnr_512", "wall_s", "nn_bytes", "effective_cr", "train_epochs"]])

In [ ]:
# ---- 绘制单模型 (single_60) 训练过程中的 PSNR 轨迹 ----
if df_cr.empty:
    raise RuntimeError("df_cr 为空：请先运行前面的 CR sweep 生成数据")

n_fields = len(TARGET_STEMS_CR)
fig, axes = plt.subplots(1, n_fields, figsize=(5.5 * n_fields, 5), dpi=140, squeeze=False)
axes = axes.ravel()

for ax, stem in zip(axes, TARGET_STEMS_CR):
    flab = FIELD_LABEL.get(stem, stem)
    
    # 筛选出当前场，并且只取 single_60 的数据（因为只有单模型存了 history）
    sub = df_cr[(df_cr["target_stem"] == stem) & (df_cr["exp"] == "single_60")]
    
    for _, row in sub.iterrows():
        # 获取你刚才加进 DataFrame 的历史记录
        h_psnr = row.get("history_psnr")
        h_time = row.get("history_time")
        
        # 确保数据存在且格式正确
        if isinstance(h_psnr, list) and isinstance(h_time, list) and len(h_psnr) > 0:
            # h_psnr 里的元素是 tuple: (epoch, psnr)，我们提取出第二个元素 psnr
            psnrs = [p[1] for p in h_psnr]
            times = h_time
            
            # 画出该 rel_err 约束下，单模型的训练时间-PSNR轨迹
            ax.plot(
                times, psnrs, 
                lw=1.5, marker=".", ms=4, 
                label=f"rel = {row['rel_err']:.0e}"
            )
            
    ax.set_title(f"{flab} — Single Model Training Curve")
    ax.set_xlabel("Cumulative Train Time (s)")
    ax.set_ylabel("Validation PSNR (dB)")
    ax.grid(True, alpha=0.35)
    
    # 防止因缺乏图例数据导致的报错
    if ax.get_legend_handles_labels()[0]:
        ax.legend(loc="lower right", fontsize=8)

fig.suptitle("Training Trajectory (single_60)", y=1.02, fontsize=13)
fig.tight_layout()
plt.show()


In [ ]:
import numpy as np
import time
import torch
import matplotlib.pyplot as plt
import concurrent.futures

# 你要对比的三个场
TARGET_STEMS_TEST = ["temperature", "dark_matter_density", "baryon_density"]
TEST_REL_ERR = 1e-4

single_time_budget = 60.0
moe_time_budget = 15.0  # 4卡并行，现实时间消耗也是 15s

fig, axes = plt.subplots(1, len(TARGET_STEMS_TEST), figsize=(6 * len(TARGET_STEMS_TEST), 5), dpi=140)
if len(TARGET_STEMS_TEST) == 1: axes = [axes]

for ax, stem in zip(axes, TARGET_STEMS_TEST):
    flab = stem.replace("_", " ").title()
    print(f"\n{'#'*60}\n# 🚀 开始评测物理场: {flab}\n{'#'*60}")
    
    # 1. 准备数据
    ctx = load_field_data(stem)
    Xps_list, _, _ = ctx["build_Xps_for_rel"](TEST_REL_ERR)
    Xps_list = [np.asarray(f, np.float32) for f in Xps_list]
    ctx["Xps_list"] = Xps_list
    X_true = ctx["Xs"][0]
    X_sz3 = Xps_list[0]
    gt_target = ctx["gt_target"]
    
    # 动态匹配 30,000 参数的深度
    try:
        h_30k, _ = pick_bg_h_under_budget(30000, shape=X_true.shape, n_fields=len(Xps_list))
        h_30k = int(h_30k)
    except Exception:
        h_30k = 20
        
    # =======================================================
    # 【基线】训练 Single Model (跑满 60s，记录完整爬升曲线)
    # =======================================================
    print(f"\n---> [Single Model] 开始串行训练 {single_time_budget}s ...")
    cfg_single = build_cfg_global(
        ctx, TEST_REL_ERR, max_train_time=single_time_budget, bg_h=h_30k, 
        steps_per_epoch=int(X_true.shape[0]), log_prefix="Single"
    )
    
    # 挂载 Evaluator 来记录每轮的全局 PSNR
    def global_evaluator(m):
        xh = run_bg_inference(m, ctx["Xs"], Xps_list, cfg_single, TEST_REL_ERR)
        return float(psnr_np(gt_target, xh))

    model_single, hist_single = train_bg_only(
        Xs=ctx["Xs"], Xps=Xps_list, device=torch.device("cuda:0"), 
        cfg=cfg_single, evaluator=global_evaluator
    )
    
    single_times = hist_single.get("time", [])
    # 取 Evaluator 返回的全局 PSNR (因为前面只返回了一个值，所以 p[1] 或者是单个 float 需要判断)
    psnr_data = hist_single.get("psnr", [])
    single_psnrs = [p[1] if isinstance(p, (tuple, list)) else p for p in psnr_data]
    
    # =======================================================
    # 【实验组】训练 Data-Driven MoE (4 卡并行，跑 15s)
    # =======================================================
    print(f"\n---> [Data-Driven MoE] 开始 4 卡并行训练 {moe_time_budget}s ...")
    err_map = np.abs(X_true - X_sz3)
    p25, p50, p75 = np.percentile(err_map, 25), np.percentile(err_map, 50), np.percentile(err_map, 75)
    masks = {
        "Exp0": err_map <= p25,
        "Exp1": (err_map > p25) & (err_map <= p50),
        "Exp2": (err_map > p50) & (err_map <= p75),
        "Exp3": err_map > p75,
    }
    
    def worker(idx, name, mask_3d):
        device = torch.device(f"cuda:{idx % torch.cuda.device_count()}")
        cfg = build_cfg_global(
            ctx, TEST_REL_ERR, max_train_time=moe_time_budget, bg_h=h_30k, 
            steps_per_epoch=int(X_true.shape[0]), log_prefix=name
        )
        cfg.bg_sampling_mask = mask_3d
        m, _ = train_bg_only(
            Xs=ctx["Xs"], Xps=ctx["Xps_list"], device=device, cfg=cfg, evaluator=None
        )
        return name, m.to("cpu"), cfg

    models_moe, cfgs_moe = {}, {}
    with concurrent.futures.ThreadPoolExecutor(max_workers=4) as executor:
        futs = [executor.submit(worker, i, n, m) for i, (n, m) in enumerate(masks.items())]
        for f in concurrent.futures.as_completed(futs):
            n, m_cpu, c = f.result()
            models_moe[n] = m_cpu
            cfgs_moe[n] = c
            
    # 并行结束，开始精准拼图
    final_recon = np.zeros_like(X_true, dtype=np.float32)
    for n, m_cpu in models_moe.items():
        m_gpu = m_cpu.to(torch.device("cuda:0"))
        pred = run_bg_inference(m_gpu, ctx["Xs"], Xps_list, cfgs_moe[n], TEST_REL_ERR)
        final_recon[masks[n]] = pred[masks[n]]
        
    moe_final_psnr = float(psnr_np(gt_target, final_recon))
    
    # =======================================================
    # 绘图逻辑
    # =======================================================
    ax.plot(single_times, single_psnrs, label=f"Single Model (-> 60s)", lw=2, marker='.', color='royalblue', alpha=0.8)
    
    # MoE 打点
    ax.scatter([moe_time_budget], [moe_final_psnr], color='crimson', marker='*', s=350, zorder=5, label=f"4-Card MoE (@15s)")
    
    # 标注具体的增益 (相比 15s 时的 Single Model)
    if len(single_times) > 0:
        idx_15s = np.argmin(np.abs(np.array(single_times) - moe_time_budget))
        psnr_single_at_15s = single_psnrs[idx_15s]
        gain = moe_final_psnr - psnr_single_at_15s
        
        # 判断是正收益还是负收益，决定颜色
        color_gain = "crimson" if gain > 0 else "green"
        sign = "+" if gain > 0 else ""
        ax.annotate(f"{sign}{gain:.2f} dB", 
                    (moe_time_budget, moe_final_psnr), 
                    xytext=(10, 10), textcoords='offset points', 
                    color=color_gain, fontweight='bold', fontsize=10)
    
    ax.set_title(f"{flab} (rel={TEST_REL_ERR:.0e})")
    ax.set_xlabel("Wall Time (s)")
    ax.set_ylabel("Global Validation PSNR (dB)")
    ax.grid(True, alpha=0.35)
    ax.legend(loc="lower right")

fig.suptitle("Performance Comparison: Single Model vs Data-Driven MoE", y=1.05, fontsize=16, fontweight='bold')
fig.tight_layout()
plt.show()
